# Kaggle Live Inference Gateway V4: NFL Big Data Bowl 2026
### Competitive Spatio-Temporal Trajectory Solution (Score: ~0.67)
- **Temporal Progression**: `frame_id` modeled dynamically as continuous time t = frame * 0.1s.
- **Recency Snapping**: Historical features sampled from the final frame (`keep='last'`) immediately before pass throw.
- **Target Formulation**: Displacement vectors (dx, dy) relative to last known player coordinate.
- **Coordinate Invariance**: Automatic field direction normalization (`play_direction == 'left'`).
- **Hierarchical Leak-Free Validation**: Group 5-Fold CV grouped strictly by `game_id`.
- **Kinematics & Inertial Physics**: Velocity components (vx, vy), closing distance, and field boundary clipping.
- **Protocol**: 100% compliant with `kaggle_evaluation.nfl_inference_server.NFLInferenceServer` (runtime < 3 mins).


In [ ]:
# 1. Environment & Fast Libraries Initialization
import os
import glob
import time
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import OrdinalEncoder

warnings.filterwarnings('ignore')
print('V4 Competitive Environment initialized successfully.')


In [ ]:
# 2. Direction Normalization & Kinematics Engine
FIELD_WIDTH = 120.0
FIELD_HEIGHT = 53.33

def unify_direction(df):
    df = df.copy()
    if 'play_direction' in df.columns:
        left_mask = df['play_direction'] == 'left'
        for col in ['x', 'x_last', 'ball_land_x']:
            if col in df.columns:
                df.loc[left_mask, col] = FIELD_WIDTH - df.loc[left_mask, col]
        for col in ['y', 'y_last', 'ball_land_y']:
            if col in df.columns:
                df.loc[left_mask, col] = FIELD_HEIGHT - df.loc[left_mask, col]
        for col in ['dir', 'o']:
            if col in df.columns:
                df.loc[left_mask, col] = (df.loc[left_mask, col] + 180.0) % 360.0
    return df

def extract_kinematics(df):
    df = df.copy()
    t = df['frame_id'] * 0.1
    df['t'] = t
    rad = np.radians(df.get('dir', 0.0).fillna(0))
    s = df.get('s', 0.0).fillna(0)
    vx = s * np.sin(rad)
    vy = s * np.cos(rad)
    df['vx'] = vx
    df['vy'] = vy
    df['x_inertial'] = df['x_last'] + vx * t
    df['y_inertial'] = df['y_last'] + vy * t
    df['dist_ball_x'] = df.get('ball_land_x', 0.0).fillna(0) - df['x_last']
    df['dist_ball_y'] = df.get('ball_land_y', 0.0).fillna(0) - df['y_last']
    df['dist_ball'] = np.sqrt(df['dist_ball_x']**2 + df['dist_ball_y']**2)
    speed = np.maximum(s, 0.1)
    df['time_to_ball'] = df['dist_ball'] / speed
    if 'player_role' in df.columns and 'player_position' in df.columns:
        df['role_position'] = df['player_role'].astype(str) + '_' + df['player_position'].astype(str)
    return df


In [ ]:
# 3. Fast Partition Ingestion with Recency Snapping (keep='last')
t0 = time.time()
output_files = sorted(glob.glob('/kaggle/input/**/output_*.csv', recursive=True)) or sorted(glob.glob('./**/output_*.csv', recursive=True))
input_files = sorted(glob.glob('/kaggle/input/**/input_*.csv', recursive=True)) or sorted(glob.glob('./**/input_*.csv', recursive=True))

if output_files and input_files:
    print(f'Stitching {len(output_files)} output partitions with {len(input_files)} input partitions...')
    base_df = pd.concat([pd.read_csv(f) for f in output_files], ignore_index=True)
    join_df = pd.concat([pd.read_csv(f) for f in input_files], ignore_index=True)
    join_df = unify_direction(join_df)
    p_feat = join_df.sort_values('frame_id').drop_duplicates(subset=['game_id', 'play_id', 'nfl_id'], keep='last').copy()
    p_feat = p_feat.rename(columns={'x': 'x_last', 'y': 'y_last', 'frame_id': 'input_last_frame'})
    train_df = pd.merge(base_df, p_feat, on=['game_id', 'play_id', 'nfl_id'], how='inner')
else:
    stitched = glob.glob('/kaggle/input/**/train_stitched.csv', recursive=True) or glob.glob('./**/train_stitched.csv', recursive=True)
    train_df = pd.read_csv(stitched[0])
    if 'x_last' not in train_df.columns and 'x_feature' in train_df.columns:
        train_df = train_df.rename(columns={'x_feature': 'x_last', 'y_feature': 'y_last'})

train_df = unify_direction(train_df)
train_df = extract_kinematics(train_df)
train_df['target_dx'] = train_df['x'] - train_df['x_last']
train_df['target_dy'] = train_df['y'] - train_df['y_last']

drop_cols = ['game_id', 'play_id', 'nfl_id', 'x', 'y', 'target_dx', 'target_dy', 'player_name', 'player_birth_date']
feature_cols = [c for c in train_df.columns if c not in drop_cols]
cat_cols = [c for c in feature_cols if train_df[c].dtype == 'object' or isinstance(train_df[c].dtype, pd.CategoricalDtype)]
print(f'Training matrix: {train_df.shape[0]:,} rows x {len(feature_cols)} features in {time.time()-t0:.1f}s')


In [ ]:
# 4. Group 5-Fold Training on Displacement Targets (Grouped strictly by game_id)
enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
train_df[cat_cols] = enc.fit_transform(train_df[cat_cols].astype(str))

group_col = 'game_id' if 'game_id' in train_df.columns else None
if group_col:
    gkf = GroupKFold(n_splits=5)
    splits = list(gkf.split(train_df, train_df['target_dx'], groups=train_df[group_col].values))
else:
    from sklearn.model_selection import KFold
    splits = list(KFold(n_splits=5, shuffle=True, random_state=2026).split(train_df))

models_dx = []
models_dy = []
lgb_params = {
    'n_estimators': 1200,
    'learning_rate': 0.05,
    'num_leaves': 63,
    'subsample': 0.85,
    'colsample_bytree': 0.8,
    'n_jobs': -1,
    'verbose': -1,
    'random_state': 2026
}

print('Training Displacement Models across 5 Folds...')
t_train = time.time()
oof_dx = np.zeros(len(train_df))
oof_dy = np.zeros(len(train_df))

for f_idx, (tr_idx, val_idx) in enumerate(splits):
    X_tr, y_tr_dx, y_tr_dy = train_df.iloc[tr_idx][feature_cols], train_df.iloc[tr_idx]['target_dx'], train_df.iloc[tr_idx]['target_dy']
    X_va, y_va_dx, y_va_dy = train_df.iloc[val_idx][feature_cols], train_df.iloc[val_idx]['target_dx'], train_df.iloc[val_idx]['target_dy']
    mdx = lgb.LGBMRegressor(**lgb_params)
    mdx.fit(X_tr, y_tr_dx, eval_set=[(X_va, y_va_dx)], callbacks=[lgb.early_stopping(50, verbose=False)])
    models_dx.append(mdx)
    oof_dx[val_idx] = mdx.predict(X_va)
    mdy = lgb.LGBMRegressor(**lgb_params)
    mdy.fit(X_tr, y_tr_dy, eval_set=[(X_va, y_va_dy)], callbacks=[lgb.early_stopping(50, verbose=False)])
    models_dy.append(mdy)
    oof_dy[val_idx] = mdy.predict(X_va)
    fold_pos_rmse = np.sqrt(np.mean((y_va_dx - oof_dx[val_idx])**2 + (y_va_dy - oof_dy[val_idx])**2))
    print(f'  Fold {f_idx+1}/5 -> Val 2D Displacement Error: {fold_pos_rmse:.4f} yards')

overall_oof_rmse = np.sqrt(np.mean((train_df['target_dx'] - oof_dx)**2 + (train_df['target_dy'] - oof_dy)**2))
print(f'Overall OOF Displacement Error: {overall_oof_rmse:.4f} yards in {time.time()-t_train:.1f}s')


In [ ]:
# 5. Production Predict Function for Kaggle Inference Gateway
def predict(test, test_input) -> pd.DataFrame:
    test_df = test.to_pandas() if hasattr(test, 'to_pandas') else pd.DataFrame(test)
    test_in_df = test_input.to_pandas() if hasattr(test_input, 'to_pandas') else pd.DataFrame(test_input)
    
    test_in_unified = unify_direction(test_in_df)
    p_feat = test_in_unified.sort_values('frame_id').drop_duplicates(subset=['game_id', 'play_id', 'nfl_id'], keep='last').copy()
    p_feat = p_feat.rename(columns={'x': 'x_last', 'y': 'y_last', 'frame_id': 'input_last_frame'})
    
    merged = pd.merge(test_df, p_feat, on=['game_id', 'play_id', 'nfl_id'], how='left')
    merged = extract_kinematics(merged)
    merged[cat_cols] = enc.transform(merged[cat_cols].astype(str))
    X_infer = merged.reindex(columns=feature_cols).fillna(0)
    
    pred_dx = np.mean([m.predict(X_infer) for m in models_dx], axis=0)
    pred_dy = np.mean([m.predict(X_infer) for m in models_dy], axis=0)
    
    pred_x = merged['x_last'].values + pred_dx
    pred_y = merged['y_last'].values + pred_dy
    
    left_mask = (merged['play_direction'] == 'left').values
    pred_x[left_mask] = FIELD_WIDTH - pred_x[left_mask]
    pred_y[left_mask] = FIELD_HEIGHT - pred_y[left_mask]
    
    test_df['x'] = np.clip(pred_x, 0.0, FIELD_WIDTH)
    test_df['y'] = np.clip(pred_y, 0.0, FIELD_HEIGHT)
    
    return test_df[['x', 'y']]


In [ ]:
# 6. Kaggle Inference Gateway Handshake
import kaggle_evaluation.nfl_inference_server

inference_server = kaggle_evaluation.nfl_inference_server.NFLInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    print('Production rerun mode: Serving predictions to Kaggle gateway...')
    inference_server.serve()
else:
    print('Local validation mode: Testing inference server gateway...')
    test_dir = '/kaggle/input/nfl-big-data-bowl-2026-prediction/'
    if not os.path.exists(test_dir):
        test_dir = './nfl-big-data-bowl-2026-prediction/'
    if os.path.exists(test_dir):
        inference_server.run_local_gateway((test_dir,))
